# TrustSafe-RAG — Exploração das consultas v0.2

Notebook somente de leitura do CSV oficial. Não gera nem modifica o dataset.

## 1. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports e carregamento do dataset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJETO = Path("/content/drive/MyDrive/TrustSafe-RAG")
ARQUIVO_CSV = PROJETO / "data" / "raw" / "consultas_mental_v02.csv"
PASTA_FIGURAS = PROJETO / "figures"
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

if not ARQUIVO_CSV.is_file():
    raise FileNotFoundError(f"CSV oficial não encontrado: {ARQUIVO_CSV}")

df = pd.read_csv(ARQUIVO_CSV, encoding="utf-8-sig")
print("Arquivo:", ARQUIVO_CSV)
print("Dimensões:", df.shape)
print("Colunas:", df.columns.tolist())
display(df.head())

## 3. Validar estrutura e distribuição

In [ ]:
obrigatorias = [
    "id", "consulta", "risco", "justificativa",
    "comportamento_esperado", "tipo_fonte_desejavel"
]
faltantes = [c for c in obrigatorias if c not in df.columns]
assert not faltantes, f"Colunas ausentes: {faltantes}"

print("Total de consultas:", len(df))
print("IDs duplicados:", df["id"].duplicated().sum())
print("\nValores ausentes:")
print(df.isna().sum())
print("\nDistribuição por risco:")
print(df["risco"].value_counts())

assert len(df) == 45
assert df["id"].is_unique
assert df["risco"].value_counts().to_dict() == {
    "baixo": 15, "moderado": 15, "alto": 15
}
print("\nValidação estrutural concluída.")

## 4. Conferir auditoria de escopo (se disponível)

In [ ]:
ARQUIVO_AUDITORIA = PROJETO / "docs" / "auditoria_escopo_v02.csv"

if ARQUIVO_AUDITORIA.is_file():
    auditoria = pd.read_csv(ARQUIVO_AUDITORIA, encoding="utf-8")
    print("Registros na auditoria:", len(auditoria))
    print("Colunas:", auditoria.columns.tolist())

    colunas = [
        "id_original", "tema", "pertence_saude_mental",
        "decisao", "justificativa", "novo_id_correspondente"
    ]
    if all(c in auditoria.columns for c in colunas):
        assert len(auditoria) == 45
        assert auditoria["id_original"].is_unique
        assert auditoria["novo_id_correspondente"].notna().all()
        assert auditoria["novo_id_correspondente"].is_unique
        assert set(auditoria["novo_id_correspondente"]) == set(df["id"])
        print("Correspondência de IDs da auditoria conferida.")
    else:
        print("A auditoria existe, mas não tem todas as colunas esperadas.")
else:
    print("Auditoria não encontrada; etapa ignorada.")

## 5. Gráfico: distribuição das classes

In [ ]:
ordem = ["baixo", "moderado", "alto"]
contagem = df["risco"].value_counts().reindex(ordem, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(contagem.index, contagem.values)
ax.set_title("Distribuição das classes de risco")
ax.set_xlabel("Classe de risco")
ax.set_ylabel("Número de consultas")
ax.bar_label(ax.containers[0])
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "distribuicao_classes.png", dpi=150)
plt.show()
plt.close(fig)

## 6. Gráfico: comprimento das consultas

In [ ]:
df["comprimento_palavras"] = df["consulta"].astype(str).apply(
    lambda texto: len(texto.split())
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["comprimento_palavras"], bins=10)
ax.set_title("Distribuição do comprimento das consultas")
ax.set_xlabel("Número de palavras")
ax.set_ylabel("Quantidade de consultas")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "comprimento_consultas.png", dpi=150)
plt.show()
plt.close(fig)

## 7. Gráfico: tipos de fontes desejáveis

In [ ]:
fontes = (
    df["tipo_fonte_desejavel"].astype(str).str.strip().value_counts().sort_values()
)
fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(fontes))))
ax.barh(fontes.index, fontes.values)
ax.set_title("Tipos de fontes desejáveis")
ax.set_xlabel("Número de consultas")
ax.set_ylabel("Tipo de fonte")
fig.tight_layout()
fig.savefig(PASTA_FIGURAS / "tipos_fontes_desejaveis.png", dpi=150)
plt.show()
plt.close(fig)

## 8. Resumo final

In [ ]:
print("Consultas analisadas:", len(df))
print("IDs únicos:", df["id"].nunique())
print("Valores ausentes:", int(df.isna().sum().sum()))
print("Figuras salvas em:", PASTA_FIGURAS)

## Observação

Este notebook não altera o CSV oficial. As figuras são recriadas na pasta `figures`.